Section 1 — Objectif

Cette partie du projet a pour objectif de réaliser une classification automatique de texte en utilisant des modèles Transformers en NLP.

Les Transformers sont utilisés car ils permettent de comprendre le contexte global d’un texte grâce au mécanisme d’attention, ce qui améliore fortement les performances par rapport aux modèles classiques.

Le dataset utilisé est basé sur des fichiers textuels contenant des exemples de phrases associées à des labels, permettant d’entraîner un modèle de classification.

 Section 2 — Dataset

Le dataset utilisé est le fichier sample_text.txt.

Ce fichier contient des données textuelles avec leurs labels associés.

Les étapes réalisées sont :

• Chargement du fichier de données
• Analyse de la structure (texte + label)
• Affichage de plusieurs exemples pour comprendre les données

Cette étape est importante pour comprendre les données avant l’entraînement du modèle.
 Section 3 — Préprocessing

Cette étape consiste à préparer les données pour le modèle.

Les opérations effectuées sont :

• Nettoyage des textes (suppression des caractères inutiles)
• Normalisation du texte
• Préparation et encodage des labels
• Séparation des données en ensemble d’entraînement et de test

 Section 4 — Tokenization

La tokenization consiste à transformer le texte en unités appelées tokens.

Les étapes incluent :

• Conversion du texte en tokens
• Application du padding pour uniformiser les séquences
• Utilisation de la truncation pour limiter la taille des textes
• Création des attention masks pour indiquer les parties importantes du texte

 Section 5 — Modèle Transformer

Un modèle Transformer pré-entraîné (BERT ou DistilBERT) est utilisé.

Les étapes sont :

• Chargement du modèle pré-entraîné
• Extraction du vecteur [CLS] représentant le texte
• Ajout d’une couche Dense pour la classification finale

 Section 6 — Compilation

Le modèle est compilé avec :

• Loss function : CrossEntropyLoss
• Optimizer : Adam
• Metrics : accuracy

Section 7 — Training

Le modèle est entraîné avec :

• 2 à 5 epochs
• Batch size adapté
• Validation sur un ensemble de test

 Pendant l’entraînement, la loss et l’accuracy sont observées pour analyser les performances.

Section 8 — Évaluation

Les performances du modèle sont analysées :

• comparaison accuracy train vs test
• détection du surapprentissage (overfitting)
• analyse des erreurs de prédiction

 Section 9 — Graphiques

Des visualisations sont utilisées pour analyser le modèle :

• courbe de loss
• courbe d’accuracy

Ces graphiques permettent de mieux comprendre l’évolution de l’entraînement.

 Section 10 — Conclusion

Cette partie a permis de mettre en place un modèle de classification de texte basé sur les Transformers.

Les résultats montrent de bonnes performances grâce au mécanisme d’attention.

Cependant, certaines limites existent comme le besoin de plus de données et le risque de surapprentissage.

Des améliorations possibles incluent l’optimisation des hyperparamètres et l’utilisation de modèles plus avancés.

In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="../data/sample_texts.txt",
    column_names=["text", "label"]
)

In [3]:
print(dataset["train"][0])

{'text': 'I love deep learning', 'label': 1}


In [4]:
print(dataset["train"][:5])

{'text': ['I love deep learning', 'Transformers are powerful models', 'NLP is fascinating', 'I hate bugs in software', 'This code is terrible'], 'label': [1, 1, 1, 0, 0]}


In [5]:
def clean(example):
    example["text"] = example["text"].lower()
    return example

dataset = dataset.map(clean)

In [6]:
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

dataset = dataset.map(tokenize, batched=True)

In [7]:
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch")

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=2   # change si ton dataset a plus de classes
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
print(dataset["train"][0])

{'text': 'i love deep learning', 'labels': tensor(1), 'input_ids': tensor([ 101, 1045, 2293, 2784, 4083,  102,    0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 0])}


In [10]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    # evaluation_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"]
)

trainer.train()

  0%|          | 0/3 [00:00<?, ?it/s]

c:\Users\adilu\anaconda3\envs\tp4_clean\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'train_runtime': 6.2735, 'train_samples_per_second': 3.347, 'train_steps_per_second': 0.478, 'train_loss': 0.6819830735524496, 'epoch': 3.0}


TrainOutput(global_step=3, training_loss=0.6819830735524496, metrics={'train_runtime': 6.2735, 'train_samples_per_second': 3.347, 'train_steps_per_second': 0.478, 'total_flos': 38032632036.0, 'train_loss': 0.6819830735524496, 'epoch': 3.0})

In [15]:
preds = trainer.predict(dataset["train"])
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print("Accuracy:", accuracy_score(y_true, y_pred))

c:\Users\adilu\anaconda3\envs\tp4_clean\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  0%|          | 0/1 [00:00<?, ?it/s]

Accuracy: 0.7142857142857143


In [13]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    return torch.argmax(outputs.logits).item()

print(predict("I love deep learning"))
print(predict("I hate bugs"))

1
1


In [14]:
texts = [
    "I love transformers",
    "This is bad",
    "Amazing work",
    "I hate this"
]

for t in texts:
    print(t, "→", predict(t))

I love transformers → 1
This is bad → 1
Amazing work → 1
I hate this → 1
